# House Prices: Advanced Regression Techniques
## Fase 1: Exploração e Limpeza de Dados

**Objetivo:** Conhecer o dataset, identificar problemas e preparar os dados para modelagem.

**Métrica:** RMSLE (Root Mean Squared Log Error)

In [ ]:
# Imports básicos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Bibliotecas carregadas!')

In [ ]:
# Carregar datasets
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f'Train: {train.shape[0]} linhas, {train.shape[1]} colunas')
print(f'Test: {test.shape[0]} linhas, {test.shape[1]} colunas')

## 1. Visão Geral dos Dados

In [ ]:
# Primeiras linhas
train.head()

In [ ]:
# Info
train.info()

In [ ]:
# Estatística descritiva
train.describe().T

In [ ]:
# Tipos de dados
print('Colunas numéricas:', train.select_dtypes(include=[np.number]).shape[1])
print('Colunas categóricas:', train.select_dtypes(include=['object']).shape[1])

# Listar colunas categóricas
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
print(f'\nColunas categóricas ({len(cat_cols)}):')
print(cat_cols)

## 2. Análise do Target (SalePrice)

In [ ]:
# Distribuição do SalePrice
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
sns.histplot(train['SalePrice'], kde=True, ax=axes[0])
axes[0].set_title('Distribuição do SalePrice')
axes[0].set_xlabel('SalePrice')

# QQ-Plot
stats.probplot(train['SalePrice'], plot=axes[1])
axes[1].set_title('QQ-Plot SalePrice')

plt.tight_layout()
plt.savefig('../reports/sale_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Estatísticas
print(f'Média: ${train["SalePrice"].mean():,.0f}')
print(f'Mediana: ${train["SalePrice"].median():,.0f}')
print(f'Desvio Padrão: ${train["SalePrice"].std():,.0f}')
print(f'Assimetria (Skew): {train["SalePrice"].skew():.3f}')
print(f'Curtose (Kurtosis): {train["SalePrice"].kurtosis():.3f}')

## 3. Valores Faltantes

In [ ]:
# Valores faltantes
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)

print(f'Colunas com valores faltantes: {len(missing)}')

missing_df = pd.DataFrame({
    'Faltantes': missing,
    '%': missing_pct
})
print(missing_df.to_string())

# Visualizar
fig, ax = plt.subplots(figsize=(12, max(6, len(missing) * 0.3)))
missing_pct.plot(kind='barh', ax=ax)
ax.set_title('Valores Faltantes por Coluna (%)')
ax.set_xlabel('% Faltante')
plt.tight_layout()
plt.savefig('../reports/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlações com SalePrice

In [ ]:
# Correlações numéricas com SalePrice
numeric_cols = train.select_dtypes(include=[np.number]).columns
correlations = train[numeric_cols].corr()['SalePrice'].sort_values(ascending=False)

print('Top 15 correlações positivas:')
print(correlations.head(15).to_string())

print('\nTop 5 correlações negativas:')
print(correlations.tail(5).to_string())

In [ ]:
# Heatmap das top correlações
top_features = correlations.head(12).index.tolist()
corr_matrix = train[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlação entre Top Features')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análise de Outliers

In [ ]:
# Scatter plot: GrLivArea vs SalePrice (relação conhecida)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=train, x='GrLivArea', y='SalePrice', ax=axes[0])
axes[0].set_title('GrLivArea vs SalePrice')

sns.scatterplot(data=train, x='TotalBsmtSF', y='SalePrice', ax=axes[1])
axes[1].set_title('TotalBsmtSF vs SalePrice')

plt.tight_layout()
plt.savefig('../reports/outliers.png', dpi=150, bbox_inches='tight')
plt.show()

# Identificar outliers óbvios
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
print(f'Outliers potenciais (GrLivArea > 4000 e SalePrice < 300k): {len(outliers)}')
if len(outliers) > 0:
    print(outliers[['Id', 'GrLivArea', 'SalePrice']].to_string())

## 6. Features Categóricas Importantes

In [ ]:
# Analisar Neighborhood vs SalePrice
fig, ax = plt.subplots(figsize=(14, 6))
order = train.groupby('Neighborhood')['SalePrice'].median().sort_values().index
sns.boxplot(data=train, x='Neighborhood', y='SalePrice', order=order, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('SalePrice por Neighborhood')
plt.tight_layout()
plt.savefig('../reports/neighborhood_price.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# OverallQual vs SalePrice
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=train, x='OverallQual', y='SalePrice', ax=ax)
ax.set_title('SalePrice por OverallQual')
plt.tight_layout()
plt.savefig('../reports/overallqual_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Resumo e Próximos Passos

### O que encontramos:
- **SalePrice** é assimétrica (skew > 0) — pode precisar de log transform
- **Valores faltantes** em várias colunas — precisamos decidir estratégia
- **Correlações fortes**: OverallQual, GrLivArea, GarageCars, TotalBsmtSF
- **Outliers** identificados em GrLivArea
- **Neighborhood** tem impacto significativo no preço

### Próximos passos (Fase 2):
1. Tratar valores faltantes
2. Feature engineering
3. Encoding de variáveis categóricas
4. Normalização do target
5. Modelo baseline